# pXRF-based geochemical differentiation

Reproduces Figure 4.14. Soil and pXRF records are merged by SSN, selected elemental concentrations are log-transformed and standardized, and PCA is used to distinguish Perkerra and Tana.

The notebook contains only the analysis retained in the final thesis. Exploratory alternatives, superseded cells, personal paths, saved outputs, and unpublished figure-formatting trials have been removed.

In [ ]:
# ============================================================
# FIGURE 4.14 — pXRF GEOCHEMICAL DIFFERENTIATION
# ------------------------------------------------------------
# Objectives:
# 1) Merge soil dataset + pXRF by SSN
# 2) Explore correlations between pXRF elements and soil variables
# 3) Run pXRF PCA to identify geochemical gradients
# 4) Test whether pXRF supports the interpretation of EC spectral
#    as an integrated soil geochemical/mineralogical signal
# ============================================================

import os
from pathlib import Path
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

from scipy.stats import spearmanr
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA

# ------------------------------------------------------------
# Global plot settings
# ------------------------------------------------------------

plt.rcParams.update({
    "font.size": 13,
    "axes.labelsize": 15,
    "axes.titlesize": 15,
    "xtick.labelsize": 13,
    "ytick.labelsize": 13,
    "legend.fontsize": 12,
    "legend.title_fontsize": 13
})

# ------------------------------------------------------------
# 1. FILE PATHS
# ------------------------------------------------------------
PROJECT_DIR = Path.cwd()
if PROJECT_DIR.name == "notebooks":
    PROJECT_DIR = PROJECT_DIR.parent

soil_file = PROJECT_DIR / "data" / "soil_data.xlsx"
pxrf_file = PROJECT_DIR / "data" / "pxrf_data.csv"
sheet_name = "PredictionNewSamplesSalinity_Fi"

out_dir = PROJECT_DIR / "outputs" / "03_pxrf_pca"
tables_dir = out_dir / "tables"
figures_dir = out_dir / "figures"

for folder in [tables_dir, figures_dir]:
    folder.mkdir(parents=True, exist_ok=True)

for required_file in [soil_file, pxrf_file]:
    if not required_file.exists():
        raise FileNotFoundError(
            f"Input file not found: {required_file}\n"
            "Place the soil and pXRF files in the data folder or update the paths."
        )

# ------------------------------------------------------------
# 2. LOAD DATA
# ------------------------------------------------------------

soil_df = pd.read_excel(soil_file, sheet_name=sheet_name)
pxrf_df = pd.read_csv(pxrf_file)

soil_df.columns = soil_df.columns.str.strip()
pxrf_df.columns = pxrf_df.columns.str.strip()

soil_df["SSN"] = soil_df["SSN"].astype(str).str.strip()
pxrf_df["SSN"] = pxrf_df["SSN"].astype(str).str.strip()

# ------------------------------------------------------------
# 3. CLEAN SOIL DATA
# ------------------------------------------------------------

soil_df["Zone"] = soil_df["SITE"].astype(str).str.strip().str.upper()
soil_df = soil_df[soil_df["Zone"].isin(["PERKERRA", "TANA"])].copy()

soil_rename = {
    "EC 1:2 (dS/cm)": "EC_spectral",
    "EC (dS/m) 1:1": "EC_lab",
    "ESP est": "ESP",
    "SOC (%)": "SOC",
    "Clay (%)": "Clay",
    "CEC cmolc/kg": "CEC",
    "TN (mg/kg)": "TN",
    "TotalBacteria": "TotalBacteria",
    "TotalFungi": "TotalFungi",
    "TYPES_BACTERIA": "TypesBacteria",
    "TYPES_FUNGI": "TypesFungi",
    "Ksat": "Ksat",
    "TotalMacrofaunaBiomass (g)": "MacrofaunaBiomass_g"
}

soil_df = soil_df.rename(columns=soil_rename)

soil_vars = [
    "Layer", "Zone", "EC_spectral", "EC_lab", "ESP", "pH",
    "SOC", "Clay", "CEC", "TN",
    "TotalBacteria", "TotalFungi", "TypesBacteria", "TypesFungi",
    "Ksat", "MacrofaunaBiomass_g"
]

soil_vars = [v for v in soil_vars if v in soil_df.columns]

for col in soil_vars:
    if col not in ["Zone"]:
        soil_df[col] = pd.to_numeric(soil_df[col], errors="coerce")

# ------------------------------------------------------------
# 4. CLEAN pXRF ELEMENTS
# ------------------------------------------------------------

element_cols_raw = [c for c in pxrf_df.columns if "(mg/kg)" in c]

for col in element_cols_raw:
    pxrf_df[col] = (
        pxrf_df[col]
        .astype(str)
        .str.replace("<LOD", "", regex=False)
        .str.strip()
    )
    pxrf_df[col] = pd.to_numeric(pxrf_df[col], errors="coerce")

pxrf_rename = {}

for col in element_cols_raw:
    clean = (
        col.replace(" (mg/kg)", "")
           .replace(" ", "_")
           .replace("/", "_")
           .replace("-", "_")
    )
    pxrf_rename[col] = f"pXRF_{clean}"

pxrf_df = pxrf_df.rename(columns=pxrf_rename)
element_cols = list(pxrf_rename.values())

# ------------------------------------------------------------
# 5. MERGE BY SSN
# ------------------------------------------------------------

pxrf_merge = pxrf_df[["SSN"] + element_cols].copy()

merged = soil_df.merge(
    pxrf_merge,
    on="SSN",
    how="left",
    validate="one_to_one"
)

merged["has_pXRF"] = merged[element_cols].notna().any(axis=1)

merge_quality = pd.DataFrame({
    "Metric": [
        "Rows soil",
        "Rows pXRF",
        "Unique SSN soil",
        "Unique SSN pXRF",
        "Rows merged",
        "Rows with pXRF"
    ],
    "Value": [
        len(soil_df),
        len(pxrf_df),
        soil_df["SSN"].nunique(),
        pxrf_df["SSN"].nunique(),
        len(merged),
        int(merged["has_pXRF"].sum())
    ]
})

print("\nMerge quality:")
print(merge_quality)

merged.to_excel(os.path.join(tables_dir, "soil_pXRF_merged.xlsx"), index=False)
merge_quality.to_excel(os.path.join(tables_dir, "merge_quality.xlsx"), index=False)

# ------------------------------------------------------------
# 6. SELECT pXRF ELEMENTS FOR ANALYSIS
# ------------------------------------------------------------

missing_pct = merged[element_cols].isna().mean() * 100
usable_elements = missing_pct[missing_pct <= 30].index.tolist()

print("\nUsable pXRF elements:")
print(usable_elements)

pd.DataFrame({
    "Element": missing_pct.index,
    "Missing_pct": missing_pct.values
}).to_excel(os.path.join(tables_dir, "pXRF_missing_summary.xlsx"), index=False)

preferred_elements = [
    "pXRF_Mg", "pXRF_Al", "pXRF_P", "pXRF_S", "pXRF_K",
    "pXRF_Ca", "pXRF_Ti", "pXRF_Mn", "pXRF_Fe",
    "pXRF_Rb", "pXRF_Sr", "pXRF_Zr", "pXRF_Ba",
    "pXRF_Th", "pXRF_U"
]

preferred_elements = [e for e in preferred_elements if e in usable_elements]

if len(preferred_elements) >= 5:
    pxrf_elements_final = preferred_elements
else:
    pxrf_elements_final = usable_elements

print("\nFinal pXRF elements used:")
print(pxrf_elements_final)

# ------------------------------------------------------------
# 8. pXRF PCA
# ------------------------------------------------------------

pca_df = merged[["SSN", "Zone", "Layer"] + pxrf_elements_final].dropna().copy()

print("\npXRF PCA dataset:", pca_df.shape)

X = pca_df[pxrf_elements_final].copy()

# Log-transform pXRF elements to reduce skewness
X_log = np.log1p(X)

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X_log)

pca = PCA()
scores = pca.fit_transform(X_scaled)

explained = pca.explained_variance_ratio_ * 100

scores_df = pd.DataFrame(
    scores,
    columns=[f"PC{i+1}" for i in range(scores.shape[1])]
)

scores_df = pd.concat(
    [pca_df[["SSN", "Zone", "Layer"]].reset_index(drop=True), scores_df],
    axis=1
)

loadings = pd.DataFrame(
    pca.components_.T,
    index=pxrf_elements_final,
    columns=[f"PC{i+1}" for i in range(len(pxrf_elements_final))]
)

explained_df = pd.DataFrame({
    "PC": [f"PC{i+1}" for i in range(len(explained))],
    "Explained_variance_pct": explained,
    "Cumulative_pct": np.cumsum(explained)
})

scores_df.to_excel(os.path.join(tables_dir, "pXRF_PCA_scores.xlsx"), index=False)
loadings.to_excel(os.path.join(tables_dir, "pXRF_PCA_loadings.xlsx"))
explained_df.to_excel(os.path.join(tables_dir, "pXRF_PCA_explained_variance.xlsx"), index=False)

print("\nExplained variance:")
print(explained_df.head(5))

# ------------------------------------------------------------
# 9. PCA FIGURE — UPDATED WITH LARGER TEXT AND MORE SEPARATION
# ------------------------------------------------------------

palette = {
    "PERKERRA": "#4C72B0",
    "TANA": "#DD8452"
}

fig, ax = plt.subplots(figsize=(10.5, 8.0))

sns.scatterplot(
    data=scores_df,
    x="PC1",
    y="PC2",
    hue="Zone",
    palette=palette,
    s=82,
    alpha=0.85,
    edgecolor="white",
    linewidth=0.7,
    ax=ax
)

# Reference axes
ax.axhline(0, color="grey", linewidth=1.0)
ax.axvline(0, color="grey", linewidth=1.0)

# Larger axis labels
ax.set_xlabel(f"PC1 ({explained[0]:.1f}%)", fontsize=16)
ax.set_ylabel(f"PC2 ({explained[1]:.1f}%)", fontsize=16)

# Larger tick labels
ax.tick_params(axis="both", labelsize=14)

# Add strongest loading arrows only
arrow_scale = 3.0

loading_strength = (
    loadings["PC1"].abs() + loadings["PC2"].abs()
).sort_values(ascending=False)

top_elements = loading_strength.head(10).index.tolist()

# ------------------------------------------------------------
# Manual offsets to separate overlapping labels
# Values are in PCA data coordinates.
# Increase/decrease these values if you want even more separation.
# ------------------------------------------------------------

label_offsets = {
    "Mg": (-0.10, -0.32),   # forced lower to avoid overlap with Ca
    "Ca": (0.07, 0.16),
    "S": (-0.08, 0.18),
    "Al": (0.13, 0.17),
    "Fe": (0.16, 0.18),
    "Rb": (0.23, 0.08),
    "Zr": (0.22, -0.04),
    "Th": (0.22, -0.17),
    "Mn": (0.22, -0.30),
    "P": (0.12, -0.15),
    "K": (0.10, 0.10),
    "Ti": (0.12, -0.08),
    "Sr": (0.12, 0.08),
    "Ba": (0.12, -0.08),
    "U": (0.12, 0.10)
}

# Larger multiplier moves all labels farther from arrow tips
label_multiplier = 1.28

for elem in top_elements:
    x = loadings.loc[elem, "PC1"] * arrow_scale
    y = loadings.loc[elem, "PC2"] * arrow_scale

    ax.arrow(
        0, 0, x, y,
        color="black",
        alpha=0.85,
        linewidth=1.4,
        head_width=0.075,
        head_length=0.10,
        length_includes_head=True
    )

    label = elem.replace("pXRF_", "")

    dx, dy = label_offsets.get(label, (0.08, 0.08))

    ax.text(
        x * label_multiplier + dx,
        y * label_multiplier + dy,
        label,
        fontsize=13,
        ha="center",
        va="center",
        bbox=dict(
            boxstyle="round,pad=0.18",
            facecolor="white",
            edgecolor="none",
            alpha=0.85
        )
    )

# Legend with larger text
ax.legend(
    title="Zone",
    title_fontsize=13,
    fontsize=12,
    frameon=False,
    loc="upper right"
)

# Add margins so labels are not too close to plot borders
ax.margins(x=0.10, y=0.12)

plt.tight_layout()

plt.savefig(
    os.path.join(figures_dir, "pXRF_PCA_zone.png"),
    dpi=600,
    bbox_inches="tight"
)

plt.savefig(
    os.path.join(figures_dir, "pXRF_PCA_zone.pdf"),
    bbox_inches="tight"
)

plt.show()
